# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Import the installed dependencies
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut
from geopy.location import Location

Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [2]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [3]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [4]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = (data.get('venue') or data.get('event') or '').strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        if data.get('latitude') is not None and data.get('longitude') is not None:
            result = Location(location, (data['latitude'], data['longitude']), {})
        else:
            result = geocoder.geocode(location, timeout=TIMEOUT)
        if result is None:
            raise ValueError(f'No coordinates found for {location}')
        location_dict[description] = result
        print(description, location_dict[description])
    except ValueError as ex:
        raise RuntimeError(f"Geocoding failed for {location}") from ex
    except GeocoderTimedOut as ex:
        raise RuntimeError(f"Geocoding timed out for {location}") from ex
    except Exception as ex:
        raise RuntimeError(f"Could not process {location}") from ex

Validity and application of temporal information extracted from patent information<br />CBI学会2025年大会 (CBI Society Annual Meeting 2025); タワーホール船堀 タワーホール船堀, 船堀街道, 船堀四丁目, 船堀, 江戸川区, 東京都, 134-0091, 日本


安全性評価手法の性能評価に向けた機械学習による対照化合物選定の検討 / Selection of reference compound list with machine learning toward validation of safety assessment methods<br />日本薬学会第144年会 (Pharmaceutical Society of Japan Annual Meeting 144); パシフィコ横浜 パシフィコ横浜, 臨港幹線道路, Minato Mirai, 高島, 西区, 横浜市, 神奈川県, 231-0017, 日本


特許情報に基づく時間軸を考慮した毒性予測の検討 / Consideration of toxicity prediction incorporating temporal information from patent data<br />第18回次世代を担う若手のための医療薬科学シンポジウム (YCPS 18); 金沢商工会議所 金沢商工会議所, お堀通り, 広坂二丁目, 西町三番丁, 金沢市, 石川県, 920-0918, 日本
特許時系列情報の医薬品への適用の妥当性とその in silico 毒性予測への応用 / Validity and application of temporal information extracted from patent information<br />第40回日本薬物動態学会（The 40th Annual Meeting of Japanese Society of Pharmacokinetics and Pharmacodynamics）; 京都市勧業館 みやこめっせ, 京都 京都市勧業館 みやこめっせ, 二条通, 岡崎最勝寺町, 左京区, 京都市, 京都府, 606-8341, 日本
Selection of reference compound list with machine learning toward validation of safety assessment methods<br />第15回日本安全性薬理研究会学術年会 (15th Annual Meeting of the Japanese Safety Pharmacology Society); 千里ライフサイエンスセンター 千里ライフサイエンスセンター


In [5]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'